# Atlas Job Setup

Run this notebook **once** to create the three Atlas pipeline jobs in Workflows.

The only thing you need to update is `NOTEBOOK_BASE_PATH` — set it to the folder where you imported the notebooks.

In [ ]:
NOTEBOOK_BASE_PATH = "/Shared/Excel_Poc"


In [ ]:
import requests

ctx      = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token    = ctx.apiToken().get()
host     = ctx.tags().get("browserHostName").get()
cluster_id = ctx.tags().get("clusterId").get()
base_url = f"https://{host}/api/2.1/jobs"
headers  = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print(f"Workspace : {host}")
print(f"Cluster   : {cluster_id}")
print()

JOBS = [
    {"name": "Job_IGTEXCEL_BRONZE", "notebook_path": f"{NOTEBOOK_BASE_PATH}/Job_IGTEXCEL_BRONZE"},
    {"name": "Job_ATLAS_SILVER",    "notebook_path": f"{NOTEBOOK_BASE_PATH}/Job_ATLAS_SILVER"},
]

created = []

for job in JOBS:
    payload = {
        "name": job["name"],
        "tasks": [
            {
                "task_key": job["name"],
                "notebook_task": {
                    "notebook_path": job["notebook_path"],
                    "source": "WORKSPACE"
                },
                "existing_cluster_id": cluster_id
            }
        ]
    }

    response = requests.post(f"{base_url}/create", headers=headers, json=payload)

    if response.status_code == 200:
        job_id  = response.json()["job_id"]
        job_url = f"https://{host}/#job/{job_id}"
        print(f"  Created : {job['name']}")
        print(f"  URL     : {job_url}")
        print()
        created.append((job["name"], job_id, job_url))
    else:
        print(f"  FAILED  : {job['name']}")
        print(f"  Error   : {response.text}")
        print()

print(f"{len(created)}/2 jobs created successfully.")
print()
print("Bookmark these URLs — use them to run the jobs:")
for name, job_id, url in created:
    print(f"  {name:30s}  {url}")
